In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [5]:
# Version 4: Self-Attention
torch.manual_seed(1337)

B, T, C = 4, 8, 32        # Batch size, block size, vocab size (each token is a vector of size 32)
x = torch.randn(B, T, C)  # Random input of shape (B, T, C)

head_size = 16
key = nn.Linear(in_features=C, out_features=head_size, bias=False)   # No bias so that solely fixed weight matrix multiplication is performed
query = nn.Linear(in_features=C, out_features=head_size, bias=False) # No bias so that solely fixed weight matrix multiplication is performed
value = nn.Linear(in_features=C, out_features=head_size, bias=False) # No bias so that solely fixed weight matrix multiplication is performed

k = key(x)   # (B, T, C) -> (B, T, head_size)
q = query(x) # (B, T, C) -> (B, T, head_size)

wei = q @ k.transpose(-2, -1)  # (B, T, head_size) @ (B, head_size, T) = (B, T, T) (T is the block_size)

tril = torch.tril(torch.ones(T, T))             # Lower triangular matrix of ones
#wei = torch.zeros((T, T))                      # (T, T)
wei = wei.masked_fill(tril == 0, float('-inf')) # Masking all values in wei where tril == 0 with -inf
wei = F.softmax(wei, dim=-1)                    # (T, T)
#out = wei @ x  # (T, T) @ (B, T, C) -> (B, T, T) @ (B, T, C) = (B, T, C)

v = value(x)   # (B, T, C) -> (B, T, head_size)
out = wei @ v  # (B, T, T) @ (B, T, head_size) = (B, T, head_size)

print(wei[0])

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)
